In [ ]:
import sys
import os

sys.path.append(os.getcwd())
# append the root directory to the sys.path

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Define Synthetic Dataset
(random - for now)

In [13]:
import numpy as np
import torch


# create synthetic dataset
class SynteticDataset:
    def __init__(self, num_samples: int= 1000, input_size: tuple= (64, 12), num_classes: int= 10):
        self.num_samples = num_samples
        self.input_size = input_size
        self.num_classes = num_classes
        self.data = self._generate_data()
    

    def _generate_data(self) -> tuple:
        X = torch.randint(0, 256, (self.num_samples, *self.input_size), dtype=torch.float32)
        y = torch.randint(0, self.num_classes, size=(self.num_samples,), dtype=torch.long)
        return X, y

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        X, y = self.data
        return X[idx], y[idx]

# define device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cuda


### Initialize simple dataloaders

In [14]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split


NUM_SAMPLES = 100
INPUT_CHANNELS = 32 # we do not have images but embeddings of image patches
INPUT_AXIS = 1 # e.g., sequence of embeddings, so 1D
INPUT_SIZE = (5, INPUT_CHANNELS) # e.g., mimicking embeddings of image patches
NUM_CLASSES = 5


dataset = SynteticDataset(num_samples=NUM_SAMPLES, input_size=INPUT_SIZE, num_classes=NUM_CLASSES)
# split train test dataset
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=2, shuffle= True)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle= False)

In [15]:
rnd_idx = np.random.randint(0, train_size)
train_sample = train_loader.dataset[rnd_idx]
print(f'Train sample shape: {train_sample[0].shape}, label: {train_sample[1]}')

Train sample shape: torch.Size([5, 32]), label: 0


In [16]:
# Initialize Perceiver model
from src.models.perceiver import Perceiver
import torch.optim as optim
import torch.nn as nn


NUM_FREQ_BANDS = 3
DEPTH = 4
MAX_FREQ = INPUT_SIZE[0]
LATENT_HEADS = 2
NUM_LATENTS = 4
LATENT_DIM = 32
CROSS_HEADS = 4
POS_ENCODING= True


model = Perceiver(num_freq_bands= NUM_FREQ_BANDS,
                  depth= DEPTH,
                  max_freq= MAX_FREQ,
                  latent_heads= LATENT_HEADS,
                  cross_heads= CROSS_HEADS,
                  input_channels= INPUT_CHANNELS,
                  input_axis= INPUT_AXIS,
                  num_classes= NUM_CLASSES,
                  fourier_encode_data= POS_ENCODING).to(device)

# Define optimizer and loss function


opti = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

### Reality check for Pereiver network on dummy data

In [17]:
def simple_train(train_loader, test_loader, device, model, criterion, optimizer, epochs= 5):
    logs = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    for epoch in range(epochs):
        model.train()
        temp_loss = 0.0
        temp_acc = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            temp_loss += loss.item() 
            temp_acc += (outputs.argmax(1) == labels).sum().item() / labels.size(0)

        epoch_loss = temp_loss / len(train_loader)
        epoch_acc = temp_acc / len(train_loader)
        print(f'Epoch {epoch+1}/{epochs}, Training Loss: {epoch_loss:.4f}, Training Accuracy: {epoch_acc:.4f}')

        # Evaluate on test set
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        print(f'Epoch {epoch+1}/{epochs}, Test Accuracy: {accuracy:.2f}%')
        logs["train_loss"].append(epoch_loss)
        logs["train_acc"].append(epoch_acc)
        logs["test_acc"].append(accuracy)
    return logs
logs = simple_train(train_loader, test_loader, device, model = model, criterion = criterion, optimizer = opti, epochs= 5)


cross attention block...
torch.cuda.FloatTensor torch.cuda.FloatTensor
cross feedforward block...
cross attention block...
torch.cuda.FloatTensor torch.cuda.FloatTensor
cross feedforward block...
cross attention block...
torch.cuda.FloatTensor torch.cuda.FloatTensor
cross feedforward block...
cross attention block...
torch.cuda.FloatTensor torch.cuda.FloatTensor
cross feedforward block...
cross attention block...
torch.cuda.FloatTensor torch.cuda.FloatTensor
cross feedforward block...
cross attention block...
torch.cuda.FloatTensor torch.cuda.FloatTensor
cross feedforward block...
cross attention block...
torch.cuda.FloatTensor torch.cuda.FloatTensor
cross feedforward block...
cross attention block...
torch.cuda.FloatTensor torch.cuda.FloatTensor
cross feedforward block...
cross attention block...
torch.cuda.FloatTensor torch.cuda.FloatTensor
cross feedforward block...
cross attention block...
torch.cuda.FloatTensor torch.cuda.FloatTensor
cross feedforward block...
cross attention bloc

KeyboardInterrupt: 

### Check with Caroline's synthetic data

In [ ]:
from src.DisentangledSSL.dataset import generate_data


args = {"dim_info": {'Y': 100, 'Z1': 50, 'Zs': 50, 'X': 100, 'Z2': 50},
        "weight_info": {'Zs': 50, 'Z1': 50, 'Z2': 50},
        "label_weight_info1": {'Zs': 0, 'Z1': 50, 'Z2': 0},
        "label_weight_info2": {'Zs': 50, 'Z1': 0, 'Z2': 0},
        "label_weight_info3": {'Zs': 0, 'Z1': 0, 'Z2': 50},
        "num_data": 1000,
        "hidden_dim": 256,
        "seed": 42}
data, targets1, targets2, targets3, X, Y, Z1, Zs, Z2 = generate_data(args["num_data"], args["hidden_dim"], args["dim_info"], args["weight_info"], args["label_weight_info1"], args["label_weight_info2"], args["label_weight_info3"], seed=args["seed"])

['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/usr/local/lib/python3.10/dist-packages', '/usr/local/lib/python3.10/dist-packages/nvfuser-0.1.1+gitunknown-py3.10-linux-x86_64.egg', '/usr/lib/python3/dist-packages', '/mnt/lts4-dislearn/scratch/home/rizou/lts4-FaP/notebooks', '/mnt/lts4-dislearn/scratch/home/rizou/lts4-FaP', '/tmp/tmp_ho72wir', '/mnt/lts4-dislearn/scratch/home/rizou/lts4-FaP', '/mnt/lts4-dislearn/scratch/home/rizou/lts4-FaP/src']
(2, 1000, 100) (1000,) (1000,) (1000,) (1000, 100) (1000, 100) (1000, 50) (1000, 50) (1000, 50)


In [ ]:
import torch

b, dim = 2, 4
s12 = torch.randn(b, dim)
s21 = torch.randn(b, dim)

features = torch.cat([s12.unsqueeze(1), s21.unsqueeze(1)], dim=1)
mask = torch.eye(b, dtype = torch.float32)
print(f"s12: {s12}  s21: {s21}")
print(f"features: {features}")
contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)

anchor_feature = contrast_feature
contrast_count = features.shape[1]
anchor_count = b
temperature = 0.07
base_temperature = 0.07

print(f"anchor_feature: {anchor_feature}")
print(f"contrast_feature: {contrast_feature}")
# compute logits
anchor_dot_contrast = torch.div(
    torch.matmul(anchor_feature, contrast_feature.T),
    temperature)

print(f"dot product: {anchor_dot_contrast}")
# for numerical stability
logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
logits = anchor_dot_contrast - logits_max.detach()

print(f"logits: {logits}")
# tile mask
mask = mask.repeat(anchor_count, contrast_count)
print(f"mask: {mask}")
logits_mask = torch.ones_like(mask)
logits_mask[:b, :b] = 0
logits_mask[b:, b:] = 0
mask = mask * logits_mask
print(f"tiled mask: {mask}")
# compute log_prob
exp_logits = torch.exp(logits) * logits_mask
log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))
print(f"log prob: {log_prob}")
# compute mean of log-likelihood over positive
mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1)

print(f"mean log prob pos: {mean_log_prob_pos}")
# loss
loss = - (temperature / base_temperature) * mean_log_prob_pos
loss = loss.view(anchor_count, b).mean()
print(f"loss: {loss}")

s12: tensor([[ 0.9438, -0.4471,  0.0437, -0.6220],
        [ 1.7637, -0.0826,  0.3861,  0.2842]])  s21: tensor([[ 0.0667,  0.9555, -0.9480,  0.8602],
        [-0.3034,  0.5900, -0.2334,  0.1839]])
features: tensor([[[ 0.9438, -0.4471,  0.0437, -0.6220],
         [ 0.0667,  0.9555, -0.9480,  0.8602]],

        [[ 1.7637, -0.0826,  0.3861,  0.2842],
         [-0.3034,  0.5900, -0.2334,  0.1839]]])
anchor_feature: tensor([[ 0.9438, -0.4471,  0.0437, -0.6220],
        [ 1.7637, -0.0826,  0.3861,  0.2842],
        [ 0.0667,  0.9555, -0.9480,  0.8602],
        [-0.3034,  0.5900, -0.2334,  0.1839]])
contrast_feature: tensor([[ 0.9438, -0.4471,  0.0437, -0.6220],
        [ 1.7637, -0.0826,  0.3861,  0.2842],
        [ 0.0667,  0.9555, -0.9480,  0.8602],
        [-0.3034,  0.5900, -0.2334,  0.1839]])
dot product: tensor([[ 21.1350,  22.0240, -13.4396,  -9.6385],
        [ 22.0240,  47.8210,  -1.1848,  -8.8819],
        [-13.4396,  -1.1848,  36.5165,  13.1849],
        [ -9.6385,  -8.8819,  13.1